# 03 · Error Analysis

Run the trained detector over the validation split, inspect false positives/negatives, and review per-class AP + tracking stability.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / 'src'))

from object_tracking_app.config.settings import get_settings
from object_tracking_app.evaluation.benchmark import run_benchmark

settings = get_settings()
export_cfg = settings.dataset.yolo_export
output_root = settings.resolve(export_cfg.output_dir)
images_dir = output_root / export_cfg.val_subdir
labels_dir = output_root / export_cfg.labels_val_subdir
print(images_dir, labels_dir)

## Run the benchmark

In [ ]:
report = run_benchmark(
    images_dir=images_dir,
    labels_dir=labels_dir,
    class_names=settings.dataset.classes,
    settings=settings,
    max_images=100,
    output_report_path=settings.resolve(settings.project_info.paths.metrics_dir) / 'error_analysis_report.json',
)
report

## Per-class AP bar chart

In [ ]:
import matplotlib.pyplot as plt

per_class = report.get('per_class_ap', {})
if per_class:
    names = list(per_class.keys())
    values = list(per_class.values())
    plt.figure(figsize=(8, 4))
    plt.bar(names, values)
    plt.xticks(rotation=45, ha='right')
    plt.ylabel('AP@0.5')
    plt.title('Per-class Average Precision')
    plt.tight_layout()
    plt.show()
else:
    print('No evaluation data -- build/train on the dataset subset first.')

## Qualitative review: worst-scoring predictions

Manually spot-check a handful of validation images with the current model to look for systematic failure patterns (e.g. small objects, occlusion, class confusion).

In [ ]:
from object_tracking_app.data.datamodule import YoloSubsetDataset
from object_tracking_app.models.detector import Detector
from object_tracking_app.utils.viz import draw_detections
import cv2
import numpy as np

dataset = YoloSubsetDataset(images_dir, labels_dir, settings.dataset.classes, img_size=settings.inference.img_size)
detector = Detector(settings=settings)

for i in range(min(3, len(dataset))):
    image, gt_boxes, gt_classes = dataset[i]
    detections = detector.predict(image, conf=settings.inference.conf_threshold)
    annotated = image.copy()
    if detections:
        boxes = np.array([d.xyxy for d in detections])
        scores = [d.confidence for d in detections]
        names = [d.class_name for d in detections]
        draw_detections(annotated, boxes, scores, names)
    plt.figure(figsize=(6, 4))
    plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    plt.title(f'Sample {i}: {len(detections)} detections, {len(gt_boxes)} ground truth')
    plt.axis('off')
    plt.show()